# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an exploration and analysis of the dataset "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(url)

# Access dataset metadata (as object, not dict)
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished if hasattr(dataset.metadata,'datePublished') else 'N/A'}")
print(f"Identifier: {dataset.metadata.identifier if hasattr(dataset.metadata, 'identifier') else 'N/A'}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all available record sets (by @id) and their fields
print("Record Sets:")
record_sets = [rs for rs in getattr(dataset.metadata, 'recordSet', [])]
if not record_sets:
    print("No record sets found in the Croissant schema. Please check the metadata.")
else:
    for i, rs in enumerate(record_sets):
        print(f"[{i}] RecordSet @id: {getattr(rs, '@id', str(rs))}")
        fields = getattr(rs, 'field', [])
        if not isinstance(fields, list):
            fields = [fields]
        for f in fields:
            fname = getattr(f, 'name', getattr(f, '@id', ''))
            print(f"   - Field: {fname} (@id={getattr(f, '@id', '')})")
    print('---\n')
    print("To see example records from a record set, use its @id in the next cell.")

In [ ]:
# (Example) Preview the first 2 records from the main record set.
# Replace <record_set_id> below with the correct @id from previous cell output.

# For this dataset, record set ids must be determined from the above cell.
# Example: Let's discover them programmatically if they exist.

record_sets = [rs for rs in getattr(dataset.metadata, 'recordSet', [])]

MAIN_RECORDSET_ID = None
for rs in record_sets:
    if hasattr(rs, '@id'):
        MAIN_RECORDSET_ID = rs['@id'] if isinstance(rs, dict) else rs.__dict__.get('@id')
        break
if MAIN_RECORDSET_ID:
    for i, x in enumerate(dataset.records(record_set=MAIN_RECORDSET_ID)):
        if i >= 2:
            break
        print(json.dumps(x, indent=2))
else:
    print("No record set found to preview records.")

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Entities are referenced via their `@id`s.

In [ ]:
# Extract all records from each record set (by @id)
record_sets = [rs for rs in getattr(dataset.metadata, 'recordSet', [])]

dataframes = {}
record_set_ids = []

for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or (rs['@id'] if isinstance(rs, dict) else None)
    if rs_id:
        record_set_ids.append(rs_id)
for rs_id in record_set_ids:
    # Load records
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"Loaded DataFrame for RecordSet: {rs_id} with {len(records)} records.")
        else:
            print(f"No records for RecordSet: {rs_id}.")
    except Exception as e:
        print(f"Could not load records for RecordSet {rs_id}: {e}")

# Display columns for first available record set with records
primary_rs_id = next(iter(dataframes.keys()), None)
if primary_rs_id:
    print(f"\nColumns in DataFrame [{primary_rs_id}]:")
    print(dataframes[primary_rs_id].columns.tolist())
    dataframes[primary_rs_id].head()
else:
    print("No dataframes were loaded; cannot show columns.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, or grouping data. All references use `@id` for record sets and fields.

In [ ]:
# Example EDA: Choose a numeric field to analyze (use @id or field name from overview)

import numpy as np

df = None
used_rs_id = None
if dataframes:
    used_rs_id = primary_rs_id 
    df = dataframes[used_rs_id].copy(deep=True)
else:
    print("No available DataFrame to analyze.")

# Try to find a likely numeric field (e.g., containing "count", "abundance", "MW", "coverage", "peptides")
numeric_candidates = []
if df is not None:
    for col in df.columns:
        if any(x in col.lower() for x in ['count', 'abundance', 'mw', 'coverage', 'peptide', 'score', 'intensity', 'num']):
            numeric_candidates.append(col)
    if not numeric_candidates:
        numeric_candidates = df.select_dtypes(include=np.number).columns.tolist()
    print(f"Numeric candidate fields: {numeric_candidates}")

# Select the first numeric candidate or specify one
numeric_field = numeric_candidates[0] if numeric_candidates else None
if numeric_field and numeric_field in df.columns:
    # Ensure numeric conversion
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to find a grouping field (e.g. a string/categorical field)
    cat_candidates = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
    group_field = cat_candidates[0] if cat_candidates else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}' (mean of {numeric_field}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

    # Pairplot across numeric candidates (if 2+ exist)
    if len(numeric_candidates) > 1 and all(x in df.columns for x in numeric_candidates):
        sns.pairplot(df[numeric_candidates].dropna())
        plt.suptitle("Pairwise Numeric Relationships", y=1.01)
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
In this notebook, you explored the FAIR\textsuperscript{2} dataset describing mass spectrometry analysis of extracellular vesicles from human mast cells. Using `mlcroissant`, you loaded the Croissant schema, inspected record sets by `@id`, extracted records for analysis, performed basic filtering, normalization, and visualized key numeric fields.

This workflow can be adapted for any Croissant-described dataset by following the same logic and updating references by entity `@id`. For further analysis, you may extend the EDA section or export processed DataFrames for downstream tasks.